# Chapter 2 — Convolutional Neural Networks

A four-module chapter for engineers who have already built dense networks
(Chapter 1) and want to understand *why* convolutions dominate computer
vision by **building and inspecting them** on real data.

Every dataset here is pulled live from the [Hugging Face Hub](https://huggingface.co/datasets)
with the `datasets` library, so you practice the same data-loading workflow used
in production ML work.

| Module | What you build | Dataset (Hugging Face) |
|---|---|---|
| 1 | 2-D convolution from scratch in NumPy: run hand-made edge detectors on real digits | `ylecun/mnist` |
| 2 | Your first CNN in Keras, head-to-head against a dense baseline | `ylecun/mnist` |
| 3 | A deeper CNN with the modern toolkit: augmentation, batch norm, dropout | `uoft-cs/cifar10` |
| 4 | X-ray vision: visualize the filters and feature maps your CNN learned | (models from 2–3) |

**How each concept is presented**, the same three passes as Chapter 1:

> 🧠 **The intuition:** the idea in plain language, no symbols.
> 📐 **The math:** the same idea written precisely, so you can read papers.
> 💻 **The code:** the same idea again, executable, in the cell that follows.

**Runtime notes** (CPU-friendly by design):
- First run downloads ~160 MB of datasets to `~/.cache/huggingface`; afterwards everything loads from disk.
- Modules 1–2 run in a couple of minutes. Module 3 trains for ~10–15 min on CPU
  (there's a knob marked `# <- knob` to shrink it), and Module 4 only inspects the model Module 3 produced.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from datasets import load_dataset
import datasets as hf_datasets

keras.utils.set_random_seed(42)            # seeds Python, NumPy, and TF in one call
rng = np.random.default_rng(seed=42)       # separate generator for our own NumPy sampling
plt.rcParams["figure.figsize"] = (7, 4.5)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| datasets", hf_datasets.__version__)

---
# Module 1 — What Convolution Actually Is

🧠 **The intuition.** A grayscale image is a 2-D array of brightness values. A
dense layer would flatten it into a long vector, throwing away the fact that
pixel $(5, 6)$ sits next to pixel $(5, 7)$, and then connect *every* input to
*every* neuron. For a 224×224 RGB photo and 1,000 neurons that is **150 million
weights** in the first layer alone.

Convolution fixes both problems with one move: instead of a weight per
(pixel, neuron) pair, take a small square of weights, a **kernel**, and slide
it across the image, taking a dot product at every stop. Two things follow, and
they are the whole reason CNNs exist. **Local connectivity** means each output
looks at a small neighborhood, matching the structure of images, because edges,
corners and textures *are* local. **Weight sharing** means the *same* kernel is
reused at every position, so an edge detector that works in the top-left works
everywhere, and a 3×3 kernel is 9 weights no matter how big the image is.

📐 **The math.**

$$ S[i, j] = \sum_{m}\sum_{n} I[i+m,\; j+n] \cdot K[m, n] $$

With a $k \times k$ kernel and no padding, an $H \times W$ image shrinks to
$(H-k+1) \times (W-k+1)$, because the kernel can't hang off the edge.

💻 **The code**, over the next five sections: load real digits, write `conv2d`
in ten lines of NumPy, run hand-designed kernels through it, and pool the
result.


## 1.1 Load MNIST from the Hugging Face Hub

`load_dataset("ylecun/mnist")` fetches the dataset the first time and caches it
locally. It returns a `DatasetDict`, a dictionary of named splits, where each
split is a table with typed columns (here: a PIL image and an integer label).


In [ ]:
mnist = load_dataset("ylecun/mnist")    # downloads once, then loads from local cache

print(mnist)                            # splits, columns, and row counts
print(mnist["train"].features)          # column types: Image + ClassLabel

sample = mnist["train"][0]              # indexing a split gives you a plain dict
print("one example ->", type(sample["image"]).__name__, "| label:", sample["label"])

## 1.2 From Hub rows to NumPy arrays

Models want dense arrays, not PIL objects. The recipe below (*shuffle, take a
slice, stack into an array*) is the standard bridge from a Hub dataset to
NumPy. We take a 20,000-image training subset to keep CPU training snappy
(swap in the full 60,000 anytime by passing `n=None`).


In [ ]:
def split_to_arrays(split, n=None, image_col="image", label_col="label", seed=42):
    """Shuffle a HF split, optionally take the first n rows, return (images, labels)."""
    ds = split.shuffle(seed=seed)                       # deterministic shuffle
    if n is not None:
        ds = ds.select(range(n))                        # cheap: just remaps indices
    images = np.stack([np.array(im) for im in ds[image_col]])   # PIL -> uint8 array per row
    labels = np.array(ds[label_col])
    return images, labels

X_train_raw, y_train = split_to_arrays(mnist["train"], n=20_000)
X_test_raw,  y_test  = split_to_arrays(mnist["test"])           # full 10k test set

print("train:", X_train_raw.shape, X_train_raw.dtype, "| test:", X_test_raw.shape)
print("pixel range:", X_train_raw.min(), "to", X_train_raw.max())

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10, 3.6))
for ax, img, label in zip(axes.flat, X_train_raw, y_train):
    ax.imshow(img, cmap="gray")
    ax.set_title(label, fontsize=10)
    ax.axis("off")
fig.suptitle("MNIST straight from the Hugging Face Hub", y=1.03)
plt.tight_layout()
plt.show()

## 1.3 Convolution in raw NumPy

💻 **The code.** Ten lines, and no libraries beyond array indexing.

*(Pedantic note: deep learning frameworks actually compute
**cross-correlation**, without flipping the kernel, and so do we. Since the
network *learns* the kernel values, the flip is irrelevant.)*


In [ ]:
def conv2d(image, kernel):
    """Slide `kernel` over `image` (no padding, stride 1) and dot-product at each stop."""
    H, W = image.shape
    kH, kW = kernel.shape
    out = np.zeros((H - kH + 1, W - kW + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            patch = image[i:i + kH, j:j + kW]     # the (kH, kW) window under the kernel
            out[i, j] = np.sum(patch * kernel)    # elementwise multiply, then sum
    return out

digit = X_train_raw[0].astype(float) / 255.0      # one real digit, scaled to [0, 1]
print("input:", digit.shape, "-> output:", conv2d(digit, np.ones((3, 3))).shape)  # 28 -> 26

## 1.4 Hand-designed kernels: the ancestors of CNNs

🧠 **The intuition.** Before 2012, computer-vision engineers designed kernels
like these *by hand*, and a "vision pipeline" was a stack of them chosen by a
person with taste. Watch what each one extracts from a real digit:

| Kernel | What it extracts |
|---|---|
| **Vertical edges** | bright where intensity changes left→right |
| **Horizontal edges** | bright where intensity changes top→bottom |
| **Sharpen** | boosts the center pixel against its neighbors |
| **Blur** | averages the neighborhood |

💻 **The code.** Same `conv2d`, four different 3×3 arrays.


In [ ]:
kernels = {
    "vertical edges":   np.array([[-1, 0, 1],
                                  [-2, 0, 2],
                                  [-1, 0, 1]], dtype=float),   # Sobel-x
    "horizontal edges": np.array([[-1, -2, -1],
                                  [ 0,  0,  0],
                                  [ 1,  2,  1]], dtype=float), # Sobel-y
    "sharpen":          np.array([[ 0, -1,  0],
                                  [-1,  5, -1],
                                  [ 0, -1,  0]], dtype=float),
    "blur":             np.ones((3, 3)) / 9.0,
}

fig, axes = plt.subplots(1, 5, figsize=(12, 2.8))
axes[0].imshow(digit, cmap="gray"); axes[0].set_title("input"); axes[0].axis("off")
for ax, (name, K) in zip(axes[1:], kernels.items()):
    ax.imshow(conv2d(digit, K), cmap="gray")
    ax.set_title(name, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 1.5 Pooling: cheap invariance

🧠 **The intuition.** A **max-pool** keeps only the strongest response in each
small window (here 2×2), halving the resolution. Two payoffs: less computation
downstream, and a little translation invariance, since if the edge shifts one
pixel the max in its window usually doesn't change.

💻 **The code.**


In [ ]:
def maxpool2(x):
    """2x2 max-pooling with stride 2: keep the max of each non-overlapping 2x2 block."""
    H, W = x.shape
    H, W = H - H % 2, W - W % 2                      # trim odd edges
    blocks = x[:H, :W].reshape(H // 2, 2, W // 2, 2) # (H/2, 2, W/2, 2)
    return blocks.max(axis=(1, 3))

edges = conv2d(digit, kernels["vertical edges"])
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, (img, title) in zip(axes, [(digit, "input 28x28"),
                                   (edges, "conv 26x26"),
                                   (maxpool2(edges), "maxpool 13x13")]):
    ax.imshow(img, cmap="gray"); ax.set_title(title); ax.axis("off")
plt.tight_layout()
plt.show()

**The punchline of Module 1:** a convolutional neural network is exactly the
pipeline you just built (convolve, nonlinearity, pool, repeat), except that the
kernel values are **learned by gradient descent** instead of designed by hand.
The network discovers its own edge detectors, then combines them into detectors
for curves, loops, and eventually whole digits.

You will see this claim verified directly in Module 4, where we open a trained
network and look at the kernels it chose.


---
# Module 2 — Your First CNN vs. a Dense Baseline

🧠 **The setup.** Same data, same optimizer, same epochs, with one dense network
and one CNN. The point is to see the difference in **parameter count** and
**accuracy** with your own eyes, on a problem simple enough that neither
architecture has an excuse.

## 2.1 Preprocess

💻 Two steps: scale pixels to $[0, 1]$, and add an explicit **channel
dimension**. Keras convolution layers expect `(batch, height, width, channels)`,
and grayscale means `channels = 1`.


In [ ]:
X_tr = (X_train_raw / 255.0).astype("float32")[..., np.newaxis]   # (20000, 28, 28, 1)
X_te = (X_test_raw  / 255.0).astype("float32")[..., np.newaxis]   # (10000, 28, 28, 1)
print("model input shape:", X_tr.shape)

## 2.2 The dense baseline

Flatten the image, push it through a hidden layer. This is the architecture you
built in Chapter 1, a solid baseline, but one that treats pixel 0 and pixel 400
as equally related.


In [ ]:
dense_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Flatten(),                        # 28*28*1 -> 784, spatial structure gone
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),  # one probability per digit class
], name="dense_baseline")

dense_model.summary()

## 2.3 The CNN

📐 **The classic small stack:** two rounds of *convolve → pool*, then classify.

- `Conv2D(32, 3)` learns **32 different 3×3 kernels**, which are 32 stacked
  "edge maps" called *feature maps*. Parameters: $32 \times (3 \cdot 3 \cdot 1) + 32$ biases $= 320$.
- `MaxPooling2D()` halves the resolution, exactly like your `maxpool2`.
- The second conv's kernels are 3×3×**32**, so they mix all 32 maps from below,
  building composite patterns out of edges.
- Only *after* the image has been distilled to compact feature maps do we
  flatten and classify.

💻 **The code.**


In [ ]:
cnn_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=3, activation="relu"),  # 28 -> 26, 32 feature maps
    layers.MaxPooling2D(),                                # 26 -> 13
    layers.Conv2D(64, kernel_size=3, activation="relu"),  # 13 -> 11, 64 feature maps
    layers.MaxPooling2D(),                                # 11 -> 5
    layers.Flatten(),                                     # 5*5*64 = 1600
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
], name="small_cnn")

cnn_model.summary()

Look at the summaries: the conv layers contribute a *tiny* fraction of the
parameters (320 + 18,496), which is weight sharing at work. Almost everything
else sits in the dense head, so the size difference between the two models comes
from something other than convolution: the CNN flattens 1,600 features where the
baseline flattens 784. The conv layers aren't expensive; they're just *better
organized* for images.

## 2.4 Train both


In [ ]:
histories = {}
for model in (dense_model, cnn_model):
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",  # labels are ints, not one-hot
                  metrics=["accuracy"])
    print(f"--- training {model.name} ---")
    histories[model.name] = model.fit(
        X_tr, y_train,
        validation_split=0.1,     # hold out 10% to watch generalization
        epochs=3,
        batch_size=128,
        verbose=2,                # one line per epoch
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, h in histories.items():
    axes[0].plot(h.history["val_accuracy"], marker="o", label=name)
    axes[1].plot(h.history["val_loss"], marker="o", label=name)
axes[0].set_title("validation accuracy"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(True)
axes[1].set_title("validation loss");     axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.show()

for name, model in [("dense", dense_model), ("cnn", cnn_model)]:
    loss, acc = model.evaluate(X_te, y_test, verbose=0)
    print(f"{name:>5} test accuracy: {acc:.4f}")

The CNN should land around **98–99%** vs roughly **97%** for the dense net, and
the gap explodes on harder data (you'll see in Module 3; on CIFAR-10 a dense net
barely reaches 45%).

## 2.5 Autopsy: what does it still get wrong?

Always look at the errors. On MNIST the survivors are usually digits humans
squint at too.


In [ ]:
probs = cnn_model.predict(X_te, verbose=0)
preds = probs.argmax(axis=1)
wrong = np.flatnonzero(preds != y_test)
print(f"{len(wrong)} mistakes out of {len(y_test)}")

show = wrong[:12]
fig, axes = plt.subplots(2, 6, figsize=(10, 3.8))
for ax, idx in zip(axes.flat, show):
    ax.imshow(X_test_raw[idx], cmap="gray")
    ax.set_title(f"true {y_test[idx]} / pred {preds[idx]}", fontsize=9)
    ax.axis("off")
fig.suptitle("The ones that got away", y=1.03)
plt.tight_layout()
plt.show()

---
# Module 3 — Real(er) Images: CIFAR-10 and the Modern Toolkit

🧠 **The intuition.** MNIST is centered, grayscale, and clean, a CNN's easiest
possible day. **CIFAR-10** is 32×32 *color* photos of ten object classes
(planes, cars, birds, cats, and six more) with messy backgrounds and wildly
varying poses. This is where dense networks fall over, at the 45% promised at
the end of Module 2, and where three standard CNN upgrades start to pay:

| Technique | What it does | Why |
|---|---|---|
| **Data augmentation** | Randomly flip/shift each training image | Free extra data; teaches invariances |
| **Batch normalization** | Re-standardizes activations inside the net | Stabilizes and speeds up training |
| **Dropout** | Randomly silences neurons during training | Fights overfitting in the dense head |

Dropout you already met in Chapter 1, Module 4, as one answer to overfitting.
The other two are new, and all three are standard equipment from here on.

## 3.1 Load CIFAR-10 from the Hub

💻 Note the small schema differences: the image column is `img`, and the class
names travel *with* the dataset via the `ClassLabel` feature. Reading the schema
instead of assuming it is a habit worth keeping.


In [ ]:
cifar = load_dataset("uoft-cs/cifar10")     # ~170 MB on first run, then cached
print(cifar["train"].features)

class_names = cifar["train"].features["label"].names
print(class_names)

In [ ]:
# One example of each class, found in the (shuffled) train split
peek = cifar["train"].shuffle(seed=0).select(range(500))
fig, axes = plt.subplots(2, 5, figsize=(10, 4.4))
for cls, ax in enumerate(axes.flat):
    row = next(r for r in peek if r["label"] == cls)
    ax.imshow(row["img"])                     # 32x32x3 RGB
    ax.set_title(class_names[cls], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3.2 To NumPy, with an honest speed/accuracy knob

`N_TRAIN = 10_000` keeps an epoch around a minute on CPU and reaches roughly
**65–70%** test accuracy. With the full 50,000 images and ~30 epochs this same
architecture reaches **~85%**, so raise the knobs if you have the patience (or a GPU).


In [ ]:
N_TRAIN = 10_000     # <- the knob: up to 50_000
EPOCHS  = 8          # <- the other knob

Xc_train_raw, yc_train = split_to_arrays(cifar["train"], n=N_TRAIN, image_col="img")
Xc_test_raw,  yc_test  = split_to_arrays(cifar["test"],  image_col="img")

Xc_tr = (Xc_train_raw / 255.0).astype("float32")   # (N, 32, 32, 3), RGB = 3 channels
Xc_te = (Xc_test_raw  / 255.0).astype("float32")
print("train:", Xc_tr.shape, "| test:", Xc_te.shape)

## 3.3 The architecture: VGG-style blocks

📐 **The pattern that generalizes far beyond this notebook:**
**[Conv → BN → ReLU] × 2 → Pool → Dropout**, with the number of filters
*doubling* each block (32 → 64 → 128) as the resolution *halves*. Spatial detail
is traded for semantic richness, stage by stage, and that trade is exactly what
Module 4 lets you watch happening.

💻 Augmentation lives *inside* the model as layers, so they perturb each batch
during `fit()` and switch themselves off at inference. We use the functional
API and name the conv layers so Module 4 can reach in and inspect them.


In [ ]:
inputs = keras.Input(shape=(32, 32, 3))

x = layers.RandomFlip("horizontal")(inputs)                 # a mirrored cat is still a cat
x = layers.RandomTranslation(0.1, 0.1)(x)                   # shift up to 10% in x and y

for block, filters in enumerate([32, 64, 128], start=1):
    for conv in (1, 2):
        x = layers.Conv2D(filters, 3, padding="same",       # "same" padding: resolution kept
                          use_bias=False,                   # BN's beta plays the bias role
                          name=f"block{block}_conv{conv}")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)                            # 32 -> 16 -> 8 -> 4
    x = layers.Dropout(0.25)(x)

x = layers.Flatten()(x)                                     # 4*4*128 = 2048
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.5)(x)                                  # heaviest dropout where params are densest
outputs = layers.Dense(10, activation="softmax")(x)

cifar_model = keras.Model(inputs, outputs, name="cifar_cnn")
cifar_model.compile(optimizer="adam",
                    loss="sparse_categorical_crossentropy",
                    metrics=["accuracy"])
cifar_model.summary()

In [ ]:
history = cifar_model.fit(
    Xc_tr, yc_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=128,
    verbose=2,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history["accuracy"], marker="o", label="train")
axes[0].plot(history.history["val_accuracy"], marker="o", label="validation")
axes[0].set_title("accuracy"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(True)
axes[1].plot(history.history["loss"], marker="o", label="train")
axes[1].plot(history.history["val_loss"], marker="o", label="validation")
axes[1].set_title("loss"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.show()

loss, acc = cifar_model.evaluate(Xc_te, yc_test, verbose=0)
print(f"CIFAR-10 test accuracy: {acc:.4f}")

Notice the curves: train accuracy may *lag* validation early on. That's the
augmentation and dropout tax: the model trains on a harder (perturbed) version of
the data than it's validated on. It's a sign the regularization is working.

## 3.4 Confusion matrix: *which* classes get confused?

Aggregate accuracy hides structure. A confusion matrix shows exactly where the
probability mass leaks.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

yc_pred = cifar_model.predict(Xc_te, verbose=0).argmax(axis=1)

fig, ax = plt.subplots(figsize=(7.5, 7))
ConfusionMatrixDisplay.from_predictions(
    yc_test, yc_pred, display_labels=class_names,
    xticks_rotation=45, colorbar=False, ax=ax, cmap="Blues",
)
ax.set_title("CIFAR-10 confusion matrix")
plt.tight_layout()
plt.show()

Typical pattern: the animal classes bleed into each other (cat ↔ dog especially)
while vehicles stay cleanly separated. The model has learned a coarse
*animal vs. machine* geometry before it fully nails the fine distinctions,
exactly what the feature-map hierarchy in Module 4 would predict.


---
# Module 4 — X-Ray Vision: What Did the Network Learn?

🧠 **The intuition.** A CNN is unusually inspectable, because its parameters
*are* pictures and its activations *are* pictures. Two dissections follow. The
**first-layer filters** are the 3×3 kernels themselves, viewable as tiny images.
The **feature maps** are what each layer outputs for a specific input, showing
the image dissolving into abstraction as it flows through the network.

## 4.1 The learned kernels of `block1_conv1`

Compare these to your hand-made Sobel kernels from Module 1. Nobody told the
network to build edge and color-contrast detectors; gradient descent decided
they were the best first move. That is the Module 1 punchline, confirmed.


In [ ]:
kernels_learned = cifar_model.get_layer("block1_conv1").kernel.numpy()  # (3, 3, 3, 32)
print("kernel tensor:", kernels_learned.shape, "-> 32 kernels of shape 3x3x3 (RGB)")

fig, axes = plt.subplots(4, 8, figsize=(9, 4.6))
for k, ax in enumerate(axes.flat):
    K = kernels_learned[:, :, :, k]
    K = (K - K.min()) / (K.max() - K.min() + 1e-9)   # normalize each kernel to [0,1] for display
    ax.imshow(K)                                      # 3 channels -> shown as an RGB micro-image
    ax.axis("off")
fig.suptitle("All 32 first-layer kernels (learned, not designed)", y=1.0)
plt.tight_layout()
plt.show()

## 4.2 Feature maps: one image's journey through the network

We build a side model that shares the CNN's layers but *returns the intermediate
outputs*. Watch the progression: early maps are recognizably the object (edges,
silhouettes); deep maps are low-resolution abstract activations that no longer
look like a picture. They are evidence like "furry texture present" or
"wheel-ish curve here".


In [ ]:
probe_layers = ["block1_conv2", "block2_conv2", "block3_conv2"]
probe = keras.Model(
    inputs=cifar_model.inputs,
    outputs=[cifar_model.get_layer(name).output for name in probe_layers],
)

img_idx = 7                                       # try other indices!
img = Xc_te[img_idx:img_idx + 1]                  # keep the batch dimension: (1, 32, 32, 3)
activations = probe.predict(img, verbose=0)

fig, axes = plt.subplots(len(probe_layers) + 1, 8, figsize=(11, 6.5))
axes[0, 0].imshow(Xc_test_raw[img_idx])
axes[0, 0].set_title(f"input: {class_names[yc_test[img_idx]]}", fontsize=9)
for ax in axes[0]:
    ax.axis("off")
for row, (name, act) in enumerate(zip(probe_layers, activations), start=1):
    strongest = act[0].mean(axis=(0, 1)).argsort()[::-1][:8]   # 8 most active channels
    for col, ch in enumerate(strongest):
        axes[row, col].imshow(act[0, :, :, ch], cmap="viridis")
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(f"{name}\n{act.shape[1]}x{act.shape[2]}",
                            rotation=0, ha="right", va="center", fontsize=9)
    axes[row, 0].axis("on"); axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
plt.tight_layout()
plt.show()

**Reading the rows:** resolution falls 32 → 16 → 8 while channel count climbs
32 → 64 → 128. The network is compressing *where* things are and expanding
*what* things are. The dense head at the end never sees pixels, and classifies
from this learned evidence vector instead.


---
# Wrap-Up

| You built | The transferable lesson |
|---|---|
| `conv2d` in NumPy | Convolution = sliding dot product; kernels are pattern detectors |
| CNN vs dense on MNIST | Locality + weight sharing beat brute-force connectivity |
| VGG-style CIFAR model | The standard block: Conv→BN→ReLU, double filters as resolution halves; augment + dropout to generalize |
| Filter & feature-map viewer | CNNs trade *where* for *what*, layer by layer |

**More Hub datasets to practice on:** `food101` (101 food classes),
`cats_vs_dogs`, `fashion_mnist`, `Bingsu/Cat_and_Dog`. The
`load_dataset → split_to_arrays → fit` workflow you now know applies to all of
them unchanged.

**Where to go next.** If convolutions are this good, why not stack fifty of
them? Through 2014 the field assumed you could. Past a certain depth, adding
layers made networks *worse*, and worse on the *training* set, so the problem
was not even overfitting. Chapter 3 measures that failure and then fixes it with
residual connections, the trick that takes CNNs from 10 layers to 100.
Chapter 4 picks up the other loose end from Module 4: those first-layer kernels
are generic, somebody else has already trained better ones on a million photos,
and you can simply take them. Object detection and segmentation reuse these
exact backbones.
